# LONG→SHORT Burst Rate Inhibition — Plots
Burst count on a LONG trial vs. RT on the immediately following SHORT trial, same style as the beta power version.

In [ ]:
import matplotlib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import statsmodels.formula.api as smf
import statsmodels.api as sm
from pathlib import Path
from scipy import stats

In [ ]:
# ── Paths & globals ───────────────────────────────────────────────────────────
data_dir = Path("/home/qmoreau/gogo_bursts/stat_output")
pc       = 12
epoch    = "STIM"

WIN_START   = 0.0
WIN_END     = 0.5
N_QUANTILES = 3
CMAP        = "PuOr"
AGE_LABELS  = ["low", "mid", "high"]

## Load trial-level LONG→SHORT burst data

In [ ]:
df = pd.read_csv(data_dir / f"PC_{pc}_{epoch}_burst_LONGtoSHORT_trials.csv")
df["response_time"] = df["response_time_ms"] / 1000.0
print(f"LONG->SHORT trials loaded: {len(df)}")
print(df[["subject_id", "age"]].drop_duplicates().shape[0], "subjects")
df.head()

## Plot 0 — Effect over time
Burst-count slope on the following SHORT trial's RT, per timepoint. Dots mark uncorrected p < 0.05 — exploratory, not cluster-corrected.

In [ ]:
trends = pd.read_csv(data_dir / f"PC_{pc}_{epoch}_burst_LONGtoSHORT_trends.csv")
trends = trends[trends["SE"].apply(lambda x: pd.notna(x) and np.isfinite(x) and x != 0)]

fig, ax = plt.subplots(1, 1, figsize=(6, 4), constrained_layout=True)

ax.axhline(0, color="k", linestyle="--", linewidth=0.8, zorder=1)
ax.axvline(0, color="k", linestyle="--", linewidth=0.8, zorder=1)

trends_sorted = trends.sort_values("time")
ax.fill_between(
    trends_sorted["time"],
    trends_sorted["estimate"] - trends_sorted["SE"],
    trends_sorted["estimate"] + trends_sorted["SE"],
    color="teal", alpha=0.15, linewidth=0,
)
ax.plot(trends_sorted["time"], trends_sorted["estimate"],
        color="teal", linewidth=1.5, label="LONG burst count -> next SHORT RT")

sig = trends_sorted[trends_sorted["p_value"] < 0.05]
ax.scatter(sig["time"], sig["estimate"], color="teal", s=18, zorder=5,
           label="p < 0.05 (uncorrected)")

ax.set_title("LONG->SHORT burst rate inhibition over time", fontweight="bold", fontsize=10)
ax.set_xlabel("Time (s)", fontsize=9)
ax.set_ylabel("Burst count slope on next-trial RT", fontsize=9)
ax.legend(fontsize=8, frameon=False)
ax.spines[["top", "right"]].set_visible(False)

out_path = data_dir / f"PC_{pc}_{epoch}_burst_LONGtoSHORT_trend_over_time.pdf"
fig.savefig(out_path, bbox_inches="tight", dpi=150)
print(f"Saved -> {out_path}")
plt.show()

## Plot 1 — Overall scatter
Mean burst count on the LONG trial vs. RT on the following SHORT trial, colored by age.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 5), constrained_layout=True)

x = df["mean_burst_count_long"].values
y = df["response_time"].values
norm  = mcolors.Normalize(vmin=df["age"].min(), vmax=df["age"].max())
cmapv = matplotlib.colormaps["viridis"]
colors = cmapv(norm(df["age"].values))

ax.scatter(x, y, c=colors, s=5, alpha=0.45, zorder=2)

slope, intercept, r, p, _ = stats.linregress(x, y)
x_line = np.linspace(x.min(), x.max(), 100)
ax.plot(x_line, intercept + slope * x_line,
        color="k", linewidth=2, zorder=4,
        label=f"r={r:.2f}, p={p:.3f}")

ax.set_title("LONG burst count → following SHORT-trial RT", fontweight="bold", fontsize=10)
ax.set_xlabel(f"Mean burst count on LONG trial [{WIN_START}, {WIN_END}] s", fontsize=9)
ax.set_ylabel("RT of following SHORT trial (s)", fontsize=9)
ax.legend(fontsize=8, frameon=False)
ax.spines[["top", "right"]].set_visible(False)

sm_scatter = cm.ScalarMappable(cmap=cmapv, norm=norm)
sm_scatter.set_array([])
cbar = fig.colorbar(sm_scatter, ax=ax, shrink=0.8, pad=0.02)
cbar.set_label("Age", fontsize=10)

fig.suptitle(
    f"Burst rate - LONG→SHORT inhibition (trial level), window [{WIN_START}, {WIN_END}] s",
    fontsize=11
)

out_path = data_dir / f"PC_{pc}_{epoch}_burst_LONGtoSHORT_scatter_trials_age_{WIN_START}_{WIN_END}.pdf"
fig.savefig(out_path, bbox_inches="tight", dpi=150)
print(f"Saved -> {out_path}")
plt.show()

## Plot 2 — Split by subject age tertile
Same relationship, fit separately per age tertile with a Gamma GLM.

In [ ]:
subj_ages = df[["subject_id", "age"]].drop_duplicates().copy()
subj_ages["age_quantile"] = pd.qcut(subj_ages["age"], q=N_QUANTILES, labels=False)
df = df.merge(subj_ages[["subject_id", "age_quantile"]], on="subject_id", how="inner")

fig, ax = plt.subplots(1, 1, figsize=(6, 5), constrained_layout=True)
x_line = np.linspace(df["mean_burst_count_long"].min(), df["mean_burst_count_long"].max(), 100)
cmap = matplotlib.colormaps[CMAP]
get_color = lambda q: cmap(0.15 + 0.7 * q / (N_QUANTILES - 1))

for q in range(N_QUANTILES):
    qdf = df[df["age_quantile"] == q].copy()
    color = get_color(q)
    age_label = AGE_LABELS[q]
    n_subj = qdf["subject_id"].nunique()
    mean_age = qdf["age"].mean()

    try:
        glm = smf.glm(
            "response_time ~ mean_burst_count_long",
            data   = qdf,
            family = sm.families.Gamma(link=sm.families.links.Log())
        ).fit()

        pred_df = glm.get_prediction(
            pd.DataFrame({"mean_burst_count_long": x_line})
        ).summary_frame(alpha=0.05)

        ax.plot(x_line, pred_df["mean"],
                color=color, linewidth=2, alpha=0.95, zorder=3,
                label=f"{age_label} ({mean_age:.0f}y, n={n_subj})")
        ax.fill_between(x_line,
                        pred_df["mean_ci_lower"],
                        pred_df["mean_ci_upper"],
                        color=color, alpha=0.12, linewidth=0)
    except Exception as e:
        print(f"  [GLM failed {age_label}]: {e}")

ax.set_title("LONG→SHORT burst rate inhibition by age tertile", fontweight="bold", fontsize=11)
ax.set_xlabel(f"Mean burst count on LONG trial [{WIN_START}, {WIN_END}] s", fontsize=9)
ax.set_ylabel("RT of following SHORT trial (s)", fontsize=9)
ax.legend(fontsize=8, frameon=False, title="Age group", title_fontsize=8)
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(
    f"Burst rate - LONG→SHORT inhibition by age tertile, window [{WIN_START}, {WIN_END}] s",
    fontsize=11
)

out_path = data_dir / f"PC_{pc}_{epoch}_burst_LONGtoSHORT_glm_age_tertiles_{WIN_START}_{WIN_END}.pdf"
fig.savefig(out_path, bbox_inches="tight", dpi=150)
print(f"Saved -> {out_path}")
plt.show()